# Banco de pruebas — OCR de actas E-14 (2ª vuelta)Explorar actas, ver los recortes y los cortes por dígito, correr cualquier modelosobre ellos y comparar ejemplares.**Cómo usarlo:** ejecutá las secciones **0** y **1** (config + utilidades) y de ahíen adelante cada sección es independiente — saltá a la que te interese.### Lo que ya está medido (para no re-descubrirlo)Sobre el **mismo** tramo de 1.500 actas no vistas (12.000–13.500):| Modelo | % de actas que cuadran ||---|---|| `digitnet.pt` (1ª vuelta, binario 28×28) | 17,5 % || gris 48×48 ronda 1 | 36,3 % || **`digitnet_2v_gris.pt` (ronda 2)** | **38,3 %** |Dos cosas que conviene no olvidar al experimentar acá:1. **El color empeora** (42,7 % RGB vs 46,9 % gris, todo lo demás igual). Usá `GRIS = True`.2. **Los tramos no son comparables entre sí**: el mismo modelo da 46,9 % en un tramo   y 36,3 % en otro. Compará siempre sobre el mismo `--desde/--limite`.3. **El techo no es 100 %**: un acta con error aritmético real *nunca* debe cuadrar.   Parte de lo que "falla" es la señal que el proyecto busca.4. **El cuello de botella son las aspas (✱)**: el modelo no tiene clase para ellas   y las lee como `7` con confianza alta → sección **5.2**. Es el arreglo con más   retorno hoy, y no se resuelve con más datos.Detalle completo en [`docs/FORENSE_COLOR.md`](../docs/FORENSE_COLOR.md).

## 0 · ConfiguraciónLo único que hace falta tocar. `MODELO` acepta cualquiera de los dos formatos(binario 28×28 de 1ª vuelta o 48×48 de 2ª) — se detecta solo.

In [ ]:
from pathlib import Path
import sys

# raíz del proyecto (funciona desde notebooks/ o desde la raíz)
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "e14").is_dir())
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "e14" / "extraccion"))

# ── qué modelo usar ──────────────────────────────────────────────────────────
MODELO = RAIZ / "models" / "digitnet_2v_gris.pt"   # ronda 2 (el mejor hoy)
# MODELO = RAIZ / "models" / "digitnet.pt"         # base de 1ª vuelta (binario)
GRIS   = True        # True salvo que quieras reproducir el experimento de color
DEV    = "cuda"      # "cpu" si no hay GPU

# ── de dónde salen las actas ─────────────────────────────────────────────────
CLAVEROS    = RAIZ / "data/segunda_vuelta/e14_pdfs_claveros"
DELEGADOS   = RAIZ / "data/segunda_vuelta/e14_pdfs_2v"
TRANSMISION = RAIZ / "data/segunda_vuelta/e14_pdfs_2v_t"

print("raíz:  ", RAIZ)
print("modelo:", MODELO.name, "| gris:", GRIS, "| dev:", DEV)
print("existe el modelo:", MODELO.exists())

## 1 · Utilidades`Lector` detecta solo el formato del checkpoint: si la primera convolución tiene1 canal de entrada es el binario 28×28 de 1ª vuelta (con su ensemble de 3), y sitiene 3 es un modelo 48×48. Así cambiar de modelo es cambiar una ruta.

In [ ]:
import numpy as np, cv2, torch, matplotlib.pyplot as plt
import posiciones_2v as P
from e14.comunes import CAND_2V
from e14.ocr.chequeo_aritmetico import chequear
from e14.ocr.dataset_color import cajas_de_acta, _normaliza_bin
from e14.ocr.clasificador_color import construir_red_color, a_gris
from e14.ocr.clasificador_digitos import construir_red as construir_red_bin

CASILLAS = [c[0] for c in P.CELDAS_2V]      # orden canónico de las 9 casillas


class Lector:
    """Envuelve cualquiera de los dos formatos de modelo tras una misma API."""

    def __init__(self, ruta, gris=True, dev="cuda"):
        self.dev = dev if torch.cuda.is_available() else "cpu"
        self.gris, self.ruta = gris, Path(ruta)
        sd = torch.load(ruta, map_location=self.dev, weights_only=False)
        sds = sd if isinstance(sd, list) else [sd]
        primera = next(v for k, v in sds[0].items() if k.endswith("weight") and v.ndim == 4)
        self.canales = primera.shape[1]              # 1 = binario 28x28, 3 = 48x48
        self.redes = []
        for s in sds:
            r = construir_red_bin() if self.canales == 1 else construir_red_color()
            r.load_state_dict(s)
            self.redes.append(r.eval().to(self.dev))
        self.tipo = "binario 28x28" if self.canales == 1 else "48x48"

    def __repr__(self):
        return f"<Lector {self.ruta.name} · {self.tipo} · ensemble={len(self.redes)} · dev={self.dev}>"

    def _tensor(self, cajas):
        if self.canales == 1:
            arr = np.stack([_normaliza_bin(c) for c in cajas])
            return torch.from_numpy(arr).float().div(255).unsqueeze(1).to(self.dev)
        arr = np.stack(cajas)
        if self.gris:
            arr = a_gris(arr)
        return torch.from_numpy(arr).permute(0, 3, 1, 2).float().div(255).to(self.dev)

    def probs(self, cajas):
        """(N,10) de probabilidades — promedio del ensemble si lo hay."""
        x = self._tensor(cajas)
        with torch.no_grad():
            p = sum(torch.softmax(r(x), 1) for r in self.redes) / len(self.redes)
        return p.cpu().numpy()


def leer(pdf, lector):
    """Lee un acta. Devuelve dict con valores, dígitos, confianzas y aritmética."""
    cajas = cajas_de_acta(pdf)
    if len(cajas) != 9:
        return None
    nombres = [n for n in CASILLAS if n in cajas]
    plano = [c for n in nombres for c in cajas[n]]
    pr = lector.probs(plano)
    dig, conf = pr.argmax(1), pr.max(1)
    out = {"pdf": Path(pdf), "cajas": cajas, "casillas": {}}
    for i, n in enumerate(nombres):
        d, c = dig[3*i:3*i+3], conf[3*i:3*i+3]
        out["casillas"][n] = {"digitos": d.tolist(), "conf": c.tolist(),
                              "valor": int("".join(map(str, d))), "conf_min": float(c.min())}
    out["valores"] = {n: v["valor"] for n, v in out["casillas"].items()}
    out["chequeo"] = chequear(out["valores"], cand=CAND_2V)
    r = out["chequeo"]
    out["cuadra"] = bool(r.cuadra_suma and r.cuadra_e11)
    return out


def actas(carpeta, desde=0, limite=None):
    """Lista de PDFs en orden estable (el mismo que usan los scripts)."""
    todos = [p for p in Path(carpeta).rglob("*.pdf") if "_logs" not in p.parts]
    return todos[desde: desde + limite if limite else None]


lector = Lector(MODELO, GRIS, DEV)
lector

## 2 · Mirar un actaCambiá `IDX` para moverte por el corpus, o pasá una ruta concreta a `pdf`.

In [ ]:
IDX = 12000                       # índice dentro de CLAVEROS
lista = actas(CLAVEROS, desde=IDX, limite=1)
pdf = lista[0]
print(pdf.relative_to(RAIZ))
print("mesa (dep,muni,zona,puesto,mesa):", P.parsear_clave(pdf))

img = P.render_pagina1(pdf, color=True)
print("página:", img.shape, "| dpi nativo:", round(P.dpi_nativo(pdf)))
plt.figure(figsize=(7, 20))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

### 2.1 · Las 9 casillas recortadasSirve para detectar desalineación de la geometría: si algún recorte cae fuera desu casilla, todo lo demás (lectura y aritmética) es basura silenciosa.

In [ ]:
celdas = P.recortar_celdas(pdf, color=True)
fig, axs = plt.subplots(len(CASILLAS), 1, figsize=(9, 1.05 * len(CASILLAS)))
for ax, nom in zip(axs, CASILLAS):
    ax.imshow(cv2.cvtColor(celdas[nom], cv2.COLOR_BGR2RGB))
    ax.set_ylabel(nom, rotation=0, ha="right", va="center", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### 2.2 · Los 27 cortes por dígito, con lo que predice el modeloCada casilla se parte en **tercios** (medido sobre 25 actas: los dígitos caencentrados en x ≈ 0,15 / 0,50 / 0,85). En rojo lo que el modelo lee con menos de`UMBRAL_CONF` de confianza — por ahí suelen empezar los errores.

In [ ]:
UMBRAL_CONF = 0.90

r = leer(pdf, lector)
fig, axs = plt.subplots(9, 3, figsize=(6, 17))
for fi, nom in enumerate(CASILLAS):
    info = r["casillas"][nom]
    for pi in range(3):
        ax = axs[fi, pi]
        ax.imshow(cv2.cvtColor(r["cajas"][nom][pi], cv2.COLOR_BGR2RGB))
        d, c = info["digitos"][pi], info["conf"][pi]
        ax.set_title(f"{d}  ({c:.2f})", fontsize=9,
                     color="crimson" if c < UMBRAL_CONF else "black")
        ax.set_xticks([]); ax.set_yticks([])
    axs[fi, 0].set_ylabel(nom, rotation=0, ha="right", va="center", fontsize=8)
plt.tight_layout(); plt.show()

### 2.3 · Lectura y aritméticaSi no cuadra, esto dice **por dónde** — y la confianza mínima ayuda a separar"el OCR falló acá" de "el acta está mal sumada de verdad", que es justo lo que elproyecto busca detectar y no hay que confundir.

In [ ]:
def resumen(r):
    q = r["chequeo"]
    print(f"{r['pdf'].name}")
    print("-" * 58)
    for nom in CASILLAS:
        info = r["casillas"][nom]
        flag = "  <-- baja confianza" if info["conf_min"] < 0.90 else ""
        print(f"  {nom:20s} {info['valor']:>4d}   conf_min={info['conf_min']:.2f}{flag}")
    print("-" * 58)
    print(f"  suma candidatos      {q.suma_candidatos}")
    print(f"  suma calculada       {q.suma_calculada}   (cand + blanco + nulo + no_marcado)")
    print(f"  SUMA_TOTAL leída     {q.suma_total_leida}   dif={q.diferencia_suma}")
    print(f"  TOTAL_E11            {q.total_e11}   dif={q.diferencia_e11}")
    print(f"\n  CUADRA: {r['cuadra']}")
    if q.notas:
        print("  notas:", q.notas)

resumen(r)

## 3 · Comparar modelos sobre la misma actaPara ver en qué dígito concreto discrepan dos modelos. Agregá o quitá rutas de`OTROS`.

In [ ]:
OTROS = [
    RAIZ / "models" / "digitnet_2v_gris.pt",
    RAIZ / "models" / "digitnet.pt",
]

lectores = []
for m in OTROS:
    if m.exists():
        lectores.append(Lector(m, GRIS, DEV))
    else:
        print("no está:", m.name)

print(f"{'casilla':20s}" + "".join(f"{l.ruta.name[:18]:>20s}" for l in lectores))
print("-" * (20 + 20 * len(lectores)))
lecturas = [leer(pdf, l) for l in lectores]
for nom in CASILLAS:
    fila = f"{nom:20s}"
    vals = [x["casillas"][nom]["valor"] for x in lecturas]
    for v in vals:
        marca = " " if len(set(vals)) == 1 else "*"
        fila += f"{str(v) + marca:>20s}"
    print(fila)
print("-" * (20 + 20 * len(lectores)))
print(f"{'CUADRA':20s}" + "".join(f"{str(x['cuadra']):>20s}" for x in lecturas))
print("\n(* = los modelos no coinciden en esa casilla)")

## 4 · La misma mesa en los tres ejemplaresAquí es donde aparecerían las enmendaduras: como **divergencia entre ejemplares**.⚠️ Dos avisos con base en lo ya medido:- Los tres ejemplares son **papeles físicos distintos**, no el mismo escaneado  tres veces. No se pueden comparar píxel a píxel.- DELEGADOS y TRANSMISIÓN están **binarizados a 1 bit** y eso *fabrica* falsos  positivos: donde el color muestra un `1` con traspaso del reverso detrás, el  umbral funde ambos en una mancha sólida. Una "divergencia" en esos dos puede  ser un artefacto del escáner, no del acta.

In [ ]:
def ruta_claveros(dep, muni, zona, puesto, mesa):
    g = list((CLAVEROS / "docs/E14" / dep / muni / zona / puesto).glob(
        f"E14_PRE_{dep}_{muni}_*_{puesto}_{int(mesa):03d}_*.pdf"))
    return g[0] if g else None


def ruta_visor(base, dep, muni, zona, puesto, mesa):
    d = base / "PRE" / dep / muni / f"{int(zona):03d}" / puesto / f"{int(mesa):03d}"
    g = list(d.glob("*.pdf"))
    return g[0] if g else None


# mesa a inspeccionar (dep, muni, zona, puesto, mesa)
DEP, MUNI, ZONA, PUESTO, MESA = "01", "280", "03", "01", "002"

fuentes = {
    "CLAVEROS":    ruta_claveros(DEP, MUNI, ZONA, PUESTO, MESA),
    "DELEGADOS":   ruta_visor(DELEGADOS, DEP, MUNI, ZONA, PUESTO, MESA),
    "TRANSMISION": ruta_visor(TRANSMISION, DEP, MUNI, ZONA, PUESTO, MESA),
}
for k, v in fuentes.items():
    print(f"{k:12s}", v.name if v else "NO ENCONTRADA")

lec = {k: leer(v, lector) for k, v in fuentes.items() if v}
print(f"\n{'casilla':20s}" + "".join(f"{k:>14s}" for k in lec))
print("-" * (20 + 14 * len(lec)))
for nom in CASILLAS:
    vals = [lec[k]["casillas"][nom]["valor"] for k in lec]
    marca = "" if len(set(vals)) == 1 else "   <-- DIVERGEN"
    print(f"{nom:20s}" + "".join(f"{v:>14d}" for v in vals) + marca)
print("-" * (20 + 14 * len(lec)))
print(f"{'CUADRA':20s}" + "".join(f"{str(lec[k]['cuadra']):>14s}" for k in lec))

### 4.1 · Ver las casillas lado a ladoComparar con el ojo antes de creerle a los números: en estos escaneos elartefacto de binarización es muy visible.

In [ ]:
QUE_CASILLAS = ["TOTAL_E11", "CANDIDATO_01", "CANDIDATO_02", "SUMA_TOTAL"]

# recortar una sola vez por fuente (recortar_celdas rasteriza la página entera)
celdas_por_fuente = {k: P.recortar_celdas(v, color=True) for k, v in fuentes.items() if v}

fig, axs = plt.subplots(len(QUE_CASILLAS), len(lec), figsize=(5 * len(lec), 1.3 * len(QUE_CASILLAS)))
for fi, nom in enumerate(QUE_CASILLAS):
    for ci, (k, r_) in enumerate(lec.items()):
        ax = axs[fi, ci] if len(QUE_CASILLAS) > 1 else axs[ci]
        ax.imshow(cv2.cvtColor(celdas_por_fuente[k][nom], cv2.COLOR_BGR2RGB))
        ax.set_title(f"{k} · {nom} = {r_['casillas'][nom]['valor']}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 5 · Correr en lote y medir el cuadreLa métrica que importa. Recordá: **compará siempre sobre el mismo tramo**(`DESDE`/`N`), porque distintos departamentos tienen distinta calidad de escaneo.

In [ ]:
DESDE, N = 12000, 300      # subí N para una medición más firme (300 ≈ 30 s en GPU)

lote = actas(CLAVEROS, desde=DESDE, limite=N)
res, fallos = [], []
for i, p in enumerate(lote, 1):
    try:
        r_ = leer(p, lector)
        if r_ is None:
            continue
        res.append(r_)
        if not r_["cuadra"]:
            fallos.append(r_)
    except Exception as e:
        pass
    if i % 100 == 0:
        print(f"  {i}/{len(lote)}  cuadre={sum(x['cuadra'] for x in res)/len(res):.1%}")

n_ok = sum(x["cuadra"] for x in res)
print(f"\nmodelo: {lector.ruta.name}   tramo: {DESDE}..{DESDE+N}")
print(f"actas leídas: {len(res)}   CUADRAN: {n_ok}  = {n_ok/max(1,len(res)):.1%}")

### 5.1 · ¿Fallo del OCR o acta genuinamente descuadrada?Esta es la distinción que decide si el proyecto encuentra algo: un acta que nocuadra puede ser un **error aritmético real** (la denuncia #1, que es señal) osimplemente una **mala lectura** (ruido). Se separan por la confianza mínima delas 27 posiciones: cuanto más confiado estuvo el modelo, más probable que eldescuadre esté en el acta y no en el OCR.Se muestra un **ranking**, no un filtro por umbral: con el modelo actualprácticamente ningún descuadre viene con las 27 lecturas confiables —lo cual yaes un resultado: **hoy el limitante sigue siendo el OCR, no hay evidencia deerrores aritméticos reales**. Cuando el modelo mejore, esta lista es por dondevan a empezar a aparecer.Es una pista para revisar a ojo, nunca un veredicto.

In [ ]:
# ranking de los descuadres por confianza de lectura (mayor = más sospechoso el ACTA)
ranking = sorted(((min(v["conf_min"] for v in r_["casillas"].values()), r_)
                  for r_ in fallos), key=lambda t: -t[0])

print(f"actas que NO cuadran: {len(fallos)} de {len(res)}")
if ranking:
    confs = np.array([c for c, _ in ranking])
    print(f"confianza mínima de esos descuadres:  p50={np.median(confs):.3f}  "
          f"p90={np.percentile(confs, 90):.3f}  máx={confs.max():.3f}")
    for u in (0.99, 0.95, 0.90):
        print(f"  con las 27 lecturas por encima de {u}: {(confs >= u).sum()}")

print("\nRanking — los de arriba son los mejores candidatos a error REAL del acta:")
for peor, r_ in ranking[:10]:
    q = r_["chequeo"]
    print(f"  {'/'.join(P.parsear_clave(r_['pdf'])):28s} conf_min={peor:.3f} "
          f"dif_suma={q.diferencia_suma} dif_e11={q.diferencia_e11}")

In [ ]:
# inspeccionar a ojo la primera del ranking
if ranking:
    r_ = ranking[0][1]
    print(r_["pdf"].relative_to(RAIZ), "\n")
    resumen(r_)
    fig, axs = plt.subplots(len(CASILLAS), 1, figsize=(9, 1.05 * len(CASILLAS)))
    celdas_ = P.recortar_celdas(r_["pdf"], color=True)
    for ax, nom in zip(axs, CASILLAS):
        ax.imshow(cv2.cvtColor(celdas_[nom], cv2.COLOR_BGR2RGB))
        ax.set_ylabel(f"{nom}\n={r_['casillas'][nom]['valor']}", rotation=0,
                      ha="right", va="center", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
else:
    print("no hubo descuadres en este lote")

### 5.2 · El fallo sistemático: las aspas de anulación**Esto es el cuello de botella actual del OCR**, encontrado con este mismo notebook.En el E-14 las casillas anuladas se marcan con un **aspa (✱)**. La regla oficial:todo-aspas = 0, y un número a la derecha del aspa es ese número (`✱84` = 84).El clasificador tiene **10 clases (0–9) y ninguna para el aspa**, así que se veforzado a inventar un dígito — y elige **7** de forma sistemática (el aspa tienetrazos diagonales), **con confianza de 0,92–0,98**. Una casilla `✱✱✱` que vale 0se lee `777`.Y hay algo peor, que explica los rendimientos decrecientes del bootstrapping:el autoetiquetado **solo conserva actas que cuadran**, o sea justamente aquellasdonde el modelo ya leía bien el aspa. Los casos donde falla nunca entran aldataset. **El autoetiquetado por aritmética es ciego a su propio errorsistemático**, y por eso no se arregla con más datos.El arreglo no es más entrenamiento: es **añadir las clases** `aspa`, `guion` y`vacío` al clasificador.

In [ ]:
# ¿cuántas casillas del lote se leen como 7xx / x7x con alta confianza?
sosp, total_casillas = [], 0
for r_ in res:
    for nom, info in r_["casillas"].items():
        total_casillas += 1
        sietes = [(d, c) for d, c in zip(info["digitos"], info["conf"]) if d == 7 and c > 0.9]
        if len(sietes) >= 2:                      # 2+ sietes seguidos = casi seguro aspas
            sosp.append((r_, nom, info))

print(f"casillas leídas: {total_casillas:,}")
print(f"con 2+ dígitos '7' de alta confianza (probables ASPAS): {len(sosp):,} "
      f"= {len(sosp)/max(1,total_casillas):.1%}")
print("\nDistribución por tipo de casilla:")
from collections import Counter
for nom, n_ in Counter(x[1] for x in sosp).most_common():
    print(f"  {nom:22s} {n_}")

Y ahora mirarlas: si son aspas, el diagnóstico está confirmado.

In [ ]:
CUANTAS = 6

fig, axs = plt.subplots(min(CUANTAS, len(sosp)), 1, figsize=(7, 1.25 * min(CUANTAS, len(sosp))))
axs = np.atleast_1d(axs)
for ax, (r_, nom, info) in zip(axs, sosp[:CUANTAS]):
    celda = P.recortar_celdas(r_["pdf"], color=True)[nom]
    ax.imshow(cv2.cvtColor(celda, cv2.COLOR_BGR2RGB))
    ax.set_ylabel(f"{nom}\nlee {info['valor']:03d}", rotation=0, ha="right",
                  va="center", fontsize=8, color="crimson")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Casillas que el modelo lee como 7 — ¿son aspas?", fontsize=10)
plt.tight_layout(); plt.show()

## 6 · Galería de dígitos por clasePara auditar el autoetiquetado y ver qué aprendió: si en la fila del `0` aparecenaspas (✱) es correcto —la regla del E-14 las cuenta como 0— pero si aparecenguiones o casillas vacías, el dataset tiene ruido.

In [ ]:
NPZ = RAIZ / "data/segunda_vuelta/digitos_2v_boot2.npz"   # el de la ronda 2

if NPZ.exists():
    d = np.load(NPZ, allow_pickle=True)
    X, y = d["X"], d["y"]
    print(f"{len(y):,} cajas · distribución: {np.bincount(y, minlength=10).tolist()}")
    POR_CLASE = 12
    fig, axs = plt.subplots(10, POR_CLASE, figsize=(POR_CLASE, 10.5))
    rng = np.random.default_rng(0)
    for c in range(10):
        idx = np.where(y == c)[0]
        sel = rng.choice(idx, min(POR_CLASE, len(idx)), replace=False) if len(idx) else []
        for j in range(POR_CLASE):
            ax = axs[c, j]; ax.set_xticks([]); ax.set_yticks([])
            if j < len(sel):
                ax.imshow(cv2.cvtColor(X[sel[j]], cv2.COLOR_BGR2RGB))
        axs[c, 0].set_ylabel(str(c), rotation=0, ha="right", va="center", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("no está el npz:", NPZ)
    print("generalo con:  python -m e14.ocr.dataset_color construir ...")

### 6.1 · Dónde se confunde el modeloMatriz de confusión sobre el propio dataset. **Va a salir casi perfecta y eso nosignifica gran cosa**: las etiquetas se generaron con un modelo hermano filtrandopor aritmética, así que mide consistencia interna, no verdad. Sirve para ver*qué pares* de dígitos se mezclan (3/8, 1/7, 4/9), no para estimar precisión.

In [ ]:
if NPZ.exists():
    rng = np.random.default_rng(1)
    sub = rng.choice(len(y), min(8000, len(y)), replace=False)
    pr = lector.probs([X[i] for i in sub])
    pred = pr.argmax(1)
    real = y[sub]
    M = np.zeros((10, 10), int)
    for a, b in zip(real, pred):
        M[a, b] += 1
    print(f"acuerdo con la etiqueta: {(pred == real).mean():.4f}   (n={len(sub)})")
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(np.log1p(M), cmap="Blues")
    ax.set_xlabel("predicho"); ax.set_ylabel("etiqueta")
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    for a in range(10):
        for b in range(10):
            if M[a, b] and a != b:
                ax.text(b, a, M[a, b], ha="center", va="center", fontsize=7, color="crimson")
    plt.colorbar(im, label="log(1+n)"); plt.title("Confusión (etiquetas autogeneradas)")
    plt.tight_layout(); plt.show()

    print("\nPares peor confundidos:")
    pares = [(M[a, b], a, b) for a in range(10) for b in range(10) if a != b]
    for n_, a, b in sorted(pares, reverse=True)[:8]:
        if n_:
            print(f"  {a} leído como {b}: {n_}")

## 7 · Buscar una mesa concretaPor códigos, en cualquiera de los tres ejemplares.

In [ ]:
def buscar(dep, muni, zona, puesto, mesa, fuente="CLAVEROS"):
    if fuente == "CLAVEROS":
        p = ruta_claveros(dep, muni, zona, puesto, mesa)
    else:
        p = ruta_visor(DELEGADOS if fuente == "DELEGADOS" else TRANSMISION,
                       dep, muni, zona, puesto, mesa)
    if not p:
        print("no encontrada"); return None
    r_ = leer(p, lector)
    resumen(r_)
    return r_


# las 4 mesas de TURBO registradas como "enmendadura reportada"
# (ojo: no hay evidencia en los datos de que lo sean — ver docs/FORENSE_COLOR.md)
for m in ("002", "003", "006", "015"):
    print("=" * 60)
    buscar("01", "280", "03", "01", m)
    print()

---## Apéndice · regenerar modelos y datasets```bash# 1) dataset autoetiquetado por aritmética (ronda 1, con el modelo de 1ª vuelta)python -m e14.ocr.dataset_color construir data/segunda_vuelta/e14_pdfs_claveros \    --salida datos_r1.npz --limite 6000 --dev cuda# 2) entrenar (SIEMPRE --gris: el color empeora)python -m e14.ocr.clasificador_color entrenar datos_r1.npz \    --modelo models/digitnet_2v_gris.pt --epochs 40 --gris# 3) medir el cuadre sobre un tramo NO usado en el entrenamientopython -m e14.ocr.clasificador_color evaluar data/segunda_vuelta/e14_pdfs_claveros \    --modelo models/digitnet_2v_gris.pt --desde 12000 --limite 1500 --gris# 4) bootstrapping: re-etiquetar con el modelo nuevo y repetir 1-3python -m e14.ocr.dataset_color construir data/segunda_vuelta/e14_pdfs_claveros \    --salida datos_r2.npz --limite 12000 --dev cuda \    --modelo-48 models/digitnet_2v_gris.pt --gris```El bootstrapping da **rendimientos decrecientes** (+2 puntos con 4,7× más datos):el cuello de botella ya no son los datos de entrenamiento.